In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

n = 20000  # Number of random combinations to try

# === 1. LOAD DATA ===
# Load Target (Ground Truth)
y_train = pd.read_csv('../data/y_train_hybrid.csv')['AVERAGE_SPEED_DIFF'].values
test_df = pd.read_csv('../data/test_data_hybrid.csv') # For RowIDs

# Load OOF (Validation) Probabilities
# Ensure these match the order of y_train!
oof_xgb = pd.read_csv('../outputs/probs/xgb_oof_probs.csv').values
oof_lgb = pd.read_csv('../outputs/probs/lgb_oof_probs.csv').values
oof_nn  = pd.read_csv('../outputs/probs/nn_oof_probs.csv').values

# Load Test Probabilities
test_xgb = pd.read_csv('../outputs/probs/xgb_test_probs.csv').values
test_lgb = pd.read_csv('../outputs/probs/lgb_test_probs.csv').values
test_nn  = pd.read_csv('../outputs/probs/nn_test_probs.csv').values

print(f"Loaded OOF Shapes: XGB{oof_xgb.shape}, LGB{oof_lgb.shape}, NN{oof_nn.shape}")

# === 2. OPTIMIZE WEIGHTS (Random Search) ===
# Random Search is more robust than Hill Climbing for small blending tasks
print(f"Optimizing weights with Random Search ({n} iterations)...")

best_score = 0
best_weights = [0.33, 0.33, 0.33]

# Try 30,000 random combinations
for i in range(n):
    # Generate 3 random numbers
    w = np.random.rand(3)
    # Normalize to sum to 1
    w = w / w.sum()
    
    # Blend OOF
    final_oof = (w[0] * oof_xgb) + (w[1] * oof_lgb) + (w[2] * oof_nn)
    preds = np.argmax(final_oof, axis=1)
    
    # Calculate Score (Macro F1)
    score = f1_score(y_train, preds, average='macro')
    
    if score > best_score:
        best_score = score
        best_weights = w

print(f"\nBEST WEIGHTS FOUND:")
print(f"XGBoost:   {best_weights[0]:.4f}")
print(f"LightGBM:  {best_weights[1]:.4f}")
print(f"NeuralNet: {best_weights[2]:.4f}")
print(f"Best OOF F1 Score: {best_score:.5f}")

# === 3. GENERATE FINAL SUBMISSION ===
print("\nGenerating Final Submission...")

# Apply Best Weights to Test Data
final_test_probs = (best_weights[0] * test_xgb) + \
                   (best_weights[1] * test_lgb) + \
                   (best_weights[2] * test_nn)

# Get Final Classes
final_preds_idx = np.argmax(final_test_probs, axis=1)
TARGET_ORDER = ['None_Existent', 'Low', 'Medium', 'High', 'Very_High']
final_labels = [TARGET_ORDER[i] for i in final_preds_idx]

# Save
submission = pd.DataFrame({
    'RowId': test_df['RowId'],
    'Speed_Diff': pd.Series(final_labels).replace('None_Existent', 'None')
})

output_path = '../outputs/submissions/submission_final_blend_V4.csv'
submission.to_csv(output_path, index=False)
print(f"'{output_path}' saved successfully!")
print(submission.head())


Loaded OOF Shapes: XGB(6812, 5), LGB(6812, 5), NN(6812, 5)
Optimizing weights with Random Search (20000 iterations)...

BEST WEIGHTS FOUND:
XGBoost:   0.3765
LightGBM:  0.2164
NeuralNet: 0.4071
Best OOF F1 Score: 0.80435

Generating Final Submission...
'../outputs/submissions/submission_final_blend_V4.csv' saved successfully!
   RowId Speed_Diff
0      1       None
1      2        Low
2      3       None
3      4       High
4      5        Low
